In [1]:
import pandas as pd
import torch
import anndata as ad
import scanpy as sc
import os
import numpy as np
import sys
script_dir='/fs/ess/PAS1475/yzhong/sf_project/codebase/utils'
sys.path.append(script_dir)
from graph_build import nodes_shift, edge_index_from_delaunay, pyg_obj, filter_edge
from scipy.spatial import Delaunay

In [2]:
file_path='/fs/ess/PAS1475/Xiaojie/graph_foundation_model'
save_path='/fs/ess/PAS1475/yzhong/sf_project/datasets/graph_noise'
os.makedirs(save_path, exist_ok=True)

In [3]:
pyg_path = "/fs/ess/PAS1475/yzhong/sf_project/backup/ccv/datasets/CosMx"
pyg_dict = torch.load(pyg_path+"/CosMx_Human_Lung_processed_scgpt_train.pt") +\
             torch.load(pyg_path+"/CosMx_Human_Lung_processed_scgpt_test.pt")
pyg_dict

/tmp/ipykernel_2307776/3751137670.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pyg_dict = torch.load(pyg_path+"/CosMx_Human_Lung_processed_scgpt_train.pt") +\
/tmp/ip

[Data(x=[98002, 512], edge_index=[2, 577056], dataset='Lung5_Rep1'),
 Data(x=[105800, 512], edge_index=[2, 623500], dataset='Lung5_Rep2'),
 Data(x=[97809, 512], edge_index=[2, 576278], dataset='Lung5_Rep3'),
 Data(x=[89975, 512], edge_index=[2, 530958], dataset='Lung6'),
 Data(x=[87606, 512], edge_index=[2, 518776], dataset='Lung9_Rep1'),
 Data(x=[71304, 512], edge_index=[2, 420238], dataset='Lung12'),
 Data(x=[139504, 512], edge_index=[2, 821692], dataset='Lung9_Rep2'),
 Data(x=[81236, 512], edge_index=[2, 479564], dataset='Lung13')]

In [4]:
adata = sc.read('/fs/ess/PAS1475/yzhong/sf_project/backup/ccv/datasets/CosMx/CosMx_Human_Lung.h5ad')
adata

AnnData object with n_obs × n_vars = 771236 × 960
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov', 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.CD298', 'Max.CD298', 'Mean.G', 'Max.G', 'Mean.Y', 'Max.Y', 'Mean.R', 'Max.R', 'Mean.DAPI', 'Max.DAPI', 'dualfiles', 'Slide_name', 'tissue', 'Run_name', 'ISH.concentration', 'Dash', 'slide_ID_numeric', 'Run_Tissue_name', 'Panel', 'Diversity', 'totalcounts', 'log10totalcounts', 'background', 'remove_flagged_cells', 'patient', 'cell_type', 'niche', 'prop_tumor_in_100_neighbors'
    var: 'feat_ID', 'remove_flagged_genes'

In [5]:
for pyg in pyg_dict:
    adata_sub = adata[adata.obs['Run_Tissue_name']==pyg.dataset].copy()
    assert adata_sub.n_obs == pyg.num_nodes, f"Number of nodes mismatch for {pyg.dataset}"

    adata_coord = adata_sub.obs[['raw_sdimx', 'raw_sdimy']]
    adata_coord_trans = adata_coord - np.min(adata_coord, axis=0)
    adata_coord_trans = adata_coord_trans * 1000
    adata_sub.obs[['X-scaled','Y-scaled']] = adata_coord_trans

    for noise_level in [0, 0.1, 0.3, 0.5]:
        adata_sub = nodes_shift(adata_sub, ['X-scaled','Y-scaled'], noise_level=noise_level)

        edge_index = edge_index_from_delaunay(filter_edge(Delaunay(adata_sub.obs[[f'X-scaled_{noise_level}_shifted',f'Y-scaled_{noise_level}_shifted']])).simplices)
        new_pyg = pyg_obj(pyg.x, edge_index)

        new_pyg.niche = torch.tensor(
            pd.Categorical(adata_sub.obs['niche']).codes,
            dtype=torch.long
        )
        new_pyg.cell_type = torch.tensor(
            pd.Categorical(adata_sub.obs['cell_type']).codes,
            dtype=torch.long
        )
        new_pyg.dataset = pyg.dataset+'_noise_level_'+str(noise_level)

        print(new_pyg)
        torch.save(
            new_pyg,
            os.path.join(
                save_path,
                f"CosMx_Lung_node_shift_noise_{noise_level}_{pyg.dataset}_scgpt.pt"
            )
        )

Data(x=[98002, 512], edge_index=[2, 577056], niche=[98002], cell_type=[98002], dataset='Lung5_Rep1_noise_level_0')
Data(x=[98002, 512], edge_index=[2, 579110], niche=[98002], cell_type=[98002], dataset='Lung5_Rep1_noise_level_0.1')
Data(x=[98002, 512], edge_index=[2, 580084], niche=[98002], cell_type=[98002], dataset='Lung5_Rep1_noise_level_0.3')
Data(x=[98002, 512], edge_index=[2, 576776], niche=[98002], cell_type=[98002], dataset='Lung5_Rep1_noise_level_0.5')
Data(x=[105800, 512], edge_index=[2, 623500], niche=[105800], cell_type=[105800], dataset='Lung5_Rep2_noise_level_0')
Data(x=[105800, 512], edge_index=[2, 626168], niche=[105800], cell_type=[105800], dataset='Lung5_Rep2_noise_level_0.1')
Data(x=[105800, 512], edge_index=[2, 626884], niche=[105800], cell_type=[105800], dataset='Lung5_Rep2_noise_level_0.3')
Data(x=[105800, 512], edge_index=[2, 623282], niche=[105800], cell_type=[105800], dataset='Lung5_Rep2_noise_level_0.5')
Data(x=[97809, 512], edge_index=[2, 576278], niche=[9780